# 06 도움말 적재 확인

ml/data/csv/의 도움말 CSV 3개(help_faq.csv, help_term.csv, help_tip.csv)를 적재 전에 검사하고, 분류와 키워드 표기를 눈으로 확인한다.

- 근거: 기능명세서 백엔드 EXT-07, 데이터 처리 기획서 3-12
- 검사는 `ml/app/help_load.py`의 `check_all`을 그대로 씀 (적재할 때와 같은 검사)
- 이 노트북은 DB를 바꾸지 않음. 적재는 4번의 명령으로
- GitHub에 올리기 전에 **출력 지우기**
- 맨 아래 **5. 확인 결과 보고서**

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# ml 폴더 찾기 (노트북 위치: ml/data/ipynb)
ML_DIR = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "app" / "help_load.py").exists())
sys.path.insert(0, str(ML_DIR))

from app.help_load import KEYWORD_COLS, check_all, value_counts, read_help

rows_by_type = read_help()
pd.DataFrame({"행 수": {t: len(rows) for t, rows in rows_by_type.items()}})

## 1. 검사 (EXT-07 예외처리)
- 번호(1 이상 숫자, 종류 안 중복), 필수 칸, 길이(제목 200, 분류 30, 키워드 각 30, 요약 200자), TIP 키워드 6개 값, 키워드 중간 빈칸, 종류별 0행
- 하나라도 걸리면 `help_load.py`는 DB에 손대지 않고 멈춤

In [ ]:
errors = check_all(rows_by_type)
print("검사 통과" if not errors else f"걸린 {len(errors)}건 (CSV를 고친 뒤 다시 실행)")
for e in errors:
    print(" ", e)

## 2. 분류, 키워드 표기 (기획서 3-12: 화면에 출력만, 오타 눈으로 확인)
- 같은 뜻인데 글자가 다르면 화면 칩이 갈라짐
- TERM 키워드 = 연관 용어, TIP 키워드 = 칩 (금리, 수수료, 상환, 계약, 불법/사기, 연체), FAQ는 키워드 안 씀

In [ ]:
for help_type, rows in rows_by_type.items():
    print(f"[{help_type}] 분류:", value_counts(rows, ["CATEGORY"]))
    print(f"[{help_type}] 키워드:", value_counts(rows, KEYWORD_COLS) or "없음")

In [ ]:
# TERM 연관 용어 중 용어 사전에 없는 단어 (화면에서 눌러도 갈 곳이 없음)
term_titles = {row["TITLE"] for row in rows_by_type["TERM"]}
print(sorted(set(value_counts(rows_by_type["TERM"], KEYWORD_COLS)) - term_titles) or "없음")

## 3. 글에 HTML 태그 모양이 있는 행
- 화면은 서버 글을 글자로만 출력함 (컴포넌트 명세서 E1). 태그를 넣으면 '<b>' 같은 글자가 그대로 보임
- 목록, 강조는 셀 안 줄바꿈(Alt+Enter)과 '1.', '•' 같은 글자로

In [ ]:
tagged = [(t, row["HELP_NO"], col) for t, rows in rows_by_type.items() for row in rows
          for col in ("TITLE", "SUMMARY", "CONTENT") if "<" in row[col] and ">" in row[col]]
pd.DataFrame(tagged, columns=["종류", "번호", "칸"]) if tagged else print("없음")

## 4. 적재 (명령 프롬프트, ml 폴더, 가상환경 켠 상태)
1. `python -m app.help_load --check`: 검사 + DB 접속 + 종류별 추가/덮어쓰기/삭제 건수 (DB 안 바꿈)
2. `python -m app.help_load`: 실제 적재 (한 트랜잭션, 번호가 같으면 덮어씀, CSV에 없는 번호는 지움, help_stat은 남김)
3. Spring 서버 다시 시작 (도움말 메모리 캐시 비움, B10)
4. 화면의 FAQ, 용어, TIP 개수 = 1번 표의 행 수

## 5. 확인 결과 보고서 (2026-11-24 실행)

### 5-1. CSV 검사

| 종류 | 행 | 분류 | 키워드 값 | 검사 |
| --- | --- | --- | --- | --- |
| FAQ | 15 | 대출 상식 7, 사이트 이용 4, 신용/금리 4 | 없음 | 통과 |
| TERM | 32 | 금리 8, 상환 7, 심사/규제 6, 기관/상품 6, 생활 금융 5 | 연관 용어 32종, 모두 사전에 있는 용어 | 통과 |
| TIP | 11 | 상환 3, 금리 2, 기관/상품 2, 심사/규제 2, 생활 금융 2 | 정한 6개 값만 | 통과 |

- HTML 태그 모양 행: 0 (FAQ 3, 5, 6, 10, 11, 13, 14의 태그를 줄바꿈과 '•', '1.' 글자로 바꾼 CSV 기준)

### 5-2. 적재 확인 (`python -m app.help_load`)

| 확인 | 결과 |
| --- | --- |
| 첫 적재 (로컬, 공용) | 추가 FAQ 15, TERM 32, TIP 11 / DB 제목, 내용이 CSV와 모두 같음 |
| 같은 CSV로 다시 적재 | help_id 그대로 (덮어쓰기 58) |
| 기획서 시험 11: 글을 고쳐 다시 적재 | 같은 help_id, help_stat 연 횟수 그대로 |
| 기획서 시험 12: TIP 1행을 빼고 다시 적재 | 그 TIP만 지워짐, help_stat 행은 남음 |
| 기획서 시험 13: 글에 '&', 작은따옴표, 줄바꿈 | 글자 그대로 들어감 |

- 시험 11부터 13은 로컬 DB에서 한 트랜잭션으로 하고 되돌림